# 08｜Fama-French 因子模型：規模、價值與動能
**資料來源：Kenneth French Data Library（達特茅斯大學）**
**理論來源：Eugene Fama & Kenneth French — Fama 2013 年諾貝爾經濟學獎得主**

> **核心問題：什麼因素決定股票的長期報酬？CAPM 說的「市場風險」夠嗎？**
>
> 本 Notebook 使用 1926 年至今的美股月資料，
> 驗證規模效應、價值效應、動能效應是否真實存在於數據中。

## 你有沒有觀察過這個現象？

台股的長期投資人常說：
「大型股穩但漲不多，小型股容易飆但要慎選。」
「帳面價值高、本益比低的股票，有時候反而比成長股更賺。」

這不是錯覺，也不是運氣。

1992 年，Eugene Fama（芝加哥大學，2013 年諾貝爾獎得主）和 Kenneth French
系統性地用幾十年的美股數據驗證了這個現象：

**小型股（市值小）和價值股（帳面市值比高）的長期報酬，確實系統性地高於大型成長股。**

但這帶來了一個問題：既然是公開的學術研究結論，市場為什麼沒有把這個套利機會消滅掉？

是因為承擔這種報酬需要付出對應的風險？還是行為偏誤讓市場持續定價錯誤？

這個辯論到今天都沒有定論——但我們可以先把數據攤開來看。

## 🎯 學習目標
1. 理解三因子模型如何改進 CAPM，解釋更多股票截面報酬差異
2. 分析 SMB（規模溢酬）和 HML（價值溢酬）的歷史表現與穩定性
3. 了解動能因子（Mom）的加入及其與三因子的關係

## ⚙️ Step 0：安裝套件

In [ ]:
%pip install pandas-datareader
print("✅ 完成")

## 匯入套件

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'Heiti TC', 'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 匯入完成")

## 一、從 CAPM 到 Fama-French 三因子

### 先說 CAPM 的故事

1964 年，William Sharpe（後來的諾貝爾獎得主）提出了一個優雅的理論：

> 你承擔的「市場風險」越多（β 越高），報酬就應該越高。

就像銀行定期存款利率比活期高，因為你犧牲了流動性；
股票波動比債券大，所以長期報酬應該比債券高——風險越大，報酬越高。

這個邏輯就是 **CAPM（資本資產定價模型）**，公式很簡單：

$$\text{股票超額報酬} = \beta \times \text{市場超額報酬}$$

只要知道一支股票的 β，你就知道它「應該」有多少報酬。

### 問題出現了

1992 年，Fama 和 French 把幾十年的美股月度數據攤開來看——
**β 不夠用。很多股票的實際報酬，CAPM 完全解釋不了。**

有兩個明顯的規律，無論怎麼調整 β 都去不掉：

1. **小型股（市值小的公司）系統性地報酬更高**——幾十年如此，不是偶然
2. **便宜股（帳面市值比高的公司）系統性地報酬更高**——就算公司本身不怎麼樣

換句話說，**市場定價股票，不只看「風險有多大」，還看「公司多大」和「現在多便宜」。**

這讓 CAPM 陷入了尷尬：要嘛承認市場不夠有效率，要嘛承認我們遺漏了某些風險因子。

### Fama-French 三因子（1993）

在市場因子之上，再加兩個：

| 因子 | 全名 | 白話說明 |
|------|------|----------|
| **Mkt-RF** | Market Risk Premium | 買整個市場比放定存多賺的那部分 |
| **SMB** | Small Minus Big | 小公司 − 大公司的報酬差（規模溢酬） |
| **HML** | High Minus Low | 便宜股 − 貴股的報酬差（價值溢酬） |

1997 年，Carhart 加入第四個：

| 因子 | 全名 | 白話說明 |
|------|------|----------|
| **Mom** | Momentum | 過去 12 個月強者 − 弱者的報酬差（動能溢酬） |

這四個因子合在一起，能解釋大多數主動基金的超額報酬——
**意味著許多「主動選股能力」，其實只是無意間押對了某個因子。**

## 二、載入 Fama-French 資料

In [ ]:
import os
import pandas_datareader.data as web

LOCAL_FF3 = "data/ff3_factors.csv"
LOCAL_MOM = "data/ff_momentum.csv"

if os.path.exists(LOCAL_FF3) and os.path.exists(LOCAL_MOM):
    ff3 = pd.read_csv(LOCAL_FF3, index_col=0, parse_dates=True)
    mom = pd.read_csv(LOCAL_MOM, index_col=0, parse_dates=True)
    print(f"✅ 從本機載入")
else:
    print("⏳ 從 Kenneth French 資料庫下載...")
    ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1926-07')[0]
    mom = web.DataReader('F-F_Momentum_Factor',       'famafrench', start='1926-07')[0]
    ff3.index = pd.to_datetime(ff3.index.to_timestamp())
    mom.index = pd.to_datetime(mom.index.to_timestamp())
    ff3.to_csv(LOCAL_FF3)
    mom.to_csv(LOCAL_MOM)
    print(f"✅ 下載完成，已快取")

# 合併，轉換為小數報酬率
factors = ff3.join(mom, how='inner') / 100
factors.columns = ['Mkt-RF', 'SMB', 'HML', 'RF', 'Mom']
factors = factors.dropna()

print(f"\n資料期間：{factors.index[0].strftime('%Y-%m')} 至 {factors.index[-1].strftime('%Y-%m')}")
print(f"總月數：{len(factors)}")
print("\n各因子月報酬統計（%）：")
print((factors[['Mkt-RF','SMB','HML','Mom']] * 100).describe().round(2).to_string())

## 三、各因子累積報酬（1926–今）

In [ ]:
cumret = (1 + factors[['Mkt-RF','SMB','HML','Mom']]).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

# 對數尺度：看長期複利效果
axes[0].semilogy(cumret.index, cumret['Mkt-RF'], label='市場超額報酬 (Mkt-RF)', linewidth=1)
axes[0].semilogy(cumret.index, cumret['SMB'],    label='規模因子 (SMB)',         linewidth=1)
axes[0].semilogy(cumret.index, cumret['HML'],    label='價值因子 (HML)',         linewidth=1)
axes[0].semilogy(cumret.index, cumret['Mom'],    label='動能因子 (Mom)',         linewidth=1)
axes[0].set_title('四大因子累積報酬（對數尺度，1 美元投入）', fontsize=12)
axes[0].set_ylabel('累積報酬（對數）')
axes[0].legend()
axes[0].axhline(1, color='black', linewidth=0.5, linestyle='--')

# 線性尺度：看各因子相對規模
axes[1].plot(cumret.index, cumret[['SMB','HML','Mom']])
axes[1].set_title('SMB / HML / Mom 累積報酬（線性尺度）', fontsize=12)
axes[1].set_ylabel('累積報酬')
axes[1].legend(['規模因子 (SMB)', '價值因子 (HML)', '動能因子 (Mom)'])
axes[1].axhline(1, color='black', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.show()

## 四、規模效應驗證（Size Premium）

**直覺問題：** 台積電這種大公司 vs 一家不知名的小工廠——你覺得哪個長期報酬更高？

大多數人說：台積電。穩、規模大、不會倒。

**數據說：小工廠。**（平均而言，而且幾十年如此。）

Fama & French 發現，**市值小的公司股票長期報酬系統性地高於大公司**。為什麼？有兩派解釋：

- **風險補償說（Fama 本人偏向這個）：** 小公司流動性差、倒閉風險高，投資人需要更高報酬才願意持有——溢酬是真實風險的補償
- **行為金融說：** 市場關注度低，分析師很少追蹤，容易被系統性地低估——是市場定價錯誤的結果

兩派都有道理。你接受哪個解釋，決定了你要不要刻意買小型股。

下面我們用數據驗證：SMB 因子（小型股 − 大型股的報酬差）是否真實存在？

In [ ]:
# 計算滾動 10 年年化 SMB 溢酬
rolling_smb = (factors['SMB'] + 1).rolling(120).apply(np.prod, raw=True) ** (1/10) - 1

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# 累積報酬
axes[0].plot(cumret.index, cumret['SMB'], color='royalblue', linewidth=1)
axes[0].axhline(1, color='black', linewidth=0.5, linestyle='--')
axes[0].fill_between(cumret.index, 1, cumret['SMB'],
                     where=(cumret['SMB'] > 1), alpha=0.2, color='green', label='小型股領先')
axes[0].fill_between(cumret.index, cumret['SMB'], 1,
                     where=(cumret['SMB'] < 1), alpha=0.2, color='red', label='大型股領先')
axes[0].set_title('規模因子（SMB）累積報酬', fontsize=12)
axes[0].legend()

# 滾動 10 年溢酬
axes[1].bar(rolling_smb.dropna().index, rolling_smb.dropna() * 100,
            color=rolling_smb.dropna().apply(lambda x: 'steelblue' if x > 0 else 'salmon'),
            width=20, alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('SMB 滾動 10 年年化溢酬（%）', fontsize=12)
axes[1].set_ylabel('年化溢酬（%）')

plt.tight_layout()
plt.show()

smb_ann = (cumret['SMB'].iloc[-1]) ** (1/len(factors) * 12) - 1
print(f"SMB 全期年化溢酬：{smb_ann*100:.2f}%")
print(f"SMB 為正（小型股領先）的月份比例：{(factors['SMB']>0).mean()*100:.1f}%")

## 五、價值效應驗證（Value Premium）

**直覺問題：** 買台積電（高 P/E、高市值、「好公司」）vs 買一家本益比 8 倍的傳統製造業——哪個報酬更高？

大多數人說：台積電。護城河、成長性、品牌。

**長期數據說：傳統製造業可能更好。**（這才是真正反直覺的地方。）

HML 因子衡量的是：高帳面市值比（便宜股、價值股）vs 低帳面市值比（成長股）的報酬差。

為什麼便宜股長期報酬更高？

> **可能是因為「便宜」本身就是風險的訊號**——公司遇到麻煩、前景不確定，
> 市場打了折扣。投資人承擔這個不確定性，所以獲得更高補償。

但 2010 年後，情況發生了變化：科技公司（低帳面市值比的成長股）開始統治市場。
Google、Apple、Nvidia——這些公司的帳面資產很少，但賺錢能力極強。

這導致 HML 因子近年表現不佳，引發學界爭論：**價值溢酬是永久性消失了，還是只是暫時低潮？**

我們用數據看歷史，你自己判斷。

In [ ]:
rolling_hml = (factors['HML'] + 1).rolling(120).apply(np.prod, raw=True) ** (1/10) - 1

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(cumret.index, cumret['HML'], color='forestgreen', linewidth=1)
axes[0].axhline(1, color='black', linewidth=0.5, linestyle='--')
axes[0].fill_between(cumret.index, 1, cumret['HML'],
                     where=(cumret['HML'] > 1), alpha=0.2, color='green', label='價值股領先')
axes[0].fill_between(cumret.index, cumret['HML'], 1,
                     where=(cumret['HML'] < 1), alpha=0.2, color='red', label='成長股領先')

# 標記 2007 年後的成長股時代
axes[0].axvspan(pd.Timestamp('2007-01'), cumret.index[-1], alpha=0.07, color='orange', label='成長股強勢期')
axes[0].set_title('價值因子（HML）累積報酬', fontsize=12)
axes[0].legend(fontsize=8)

axes[1].bar(rolling_hml.dropna().index, rolling_hml.dropna() * 100,
            color=rolling_hml.dropna().apply(lambda x: 'forestgreen' if x > 0 else 'salmon'),
            width=20, alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('HML 滾動 10 年年化溢酬（%）', fontsize=12)
axes[1].set_ylabel('年化溢酬（%）')

plt.tight_layout()
plt.show()

hml_ann = (cumret['HML'].iloc[-1]) ** (1/len(factors) * 12) - 1
print(f"HML 全期年化溢酬：{hml_ann*100:.2f}%")
recent = factors.loc['2010':, 'HML']
print(f"2010 年後 HML 月均報酬：{recent.mean()*100:.3f}%（{'正' if recent.mean()>0 else '負'}溢酬）")

## 六、動能效應驗證（Momentum）

**直覺問題：** 某支股票漲了六個月，你敢買嗎？

台灣散戶常說：「漲多了，不敢追。萬一追高怎麼辦？」
這個直覺叫做「均值回歸」——漲多的遲早會跌回來。

**但統計說：短期方向上，強者恆強。**

動能效應（Momentum）：**過去 3–12 個月報酬排名高的股票，接下來 1 個月繼續領先的機率，統計上顯著高於平均。**

這和「均值回歸」互相矛盾嗎？不——它們只是在不同的時間尺度上成立：

| 時間框架 | 現象 | 方向 |
|----------|------|------|
| 3–12 個月 | **動能效應**：強者繼續強 | 順勢 |
| 3–5 年 | **均值回歸**：贏家終將回落 | 逆勢 |

**動能因子有一個致命弱點：Momentum Crash（動能崩潰）**

在市場急速反轉的時候（例如 2009 年 3 月、1932 年）——
之前的輸家突然大反彈，之前的贏家反而崩跌。
動能策略在這種情況下會在極短時間（幾週內）損失 30–50%。

這不是理論風險，是歷史上真實發生過的事。我們在數據裡可以看到。

In [ ]:
rolling_mom = (factors['Mom'] + 1).rolling(120).apply(np.prod, raw=True) ** (1/10) - 1

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(cumret.index, cumret['Mom'], color='darkorange', linewidth=1)
axes[0].axhline(1, color='black', linewidth=0.5, linestyle='--')
axes[0].fill_between(cumret.index, 1, cumret['Mom'],
                     where=(cumret['Mom'] > 1), alpha=0.2, color='green', label='動能策略獲利')
axes[0].fill_between(cumret.index, cumret['Mom'], 1,
                     where=(cumret['Mom'] < 1), alpha=0.2, color='red', label='動能策略虧損')

# 標記動能崩潰（Momentum Crash）
crashes = {'1932\n崩潰': '1932-08', '2009\n崩潰': '2009-03'}
for label, d in crashes.items():
    date = pd.to_datetime(d)
    y = cumret['Mom'].asof(date)
    axes[0].annotate(label, xy=(date, y), xytext=(date, y * 1.5),
                     fontsize=8, ha='center', color='darkred',
                     arrowprops=dict(arrowstyle='->', color='darkred', lw=0.8))

axes[0].set_title('動能因子（Mom）累積報酬', fontsize=12)
axes[0].legend(fontsize=8)

axes[1].bar(rolling_mom.dropna().index, rolling_mom.dropna() * 100,
            color=rolling_mom.dropna().apply(lambda x: 'darkorange' if x > 0 else 'salmon'),
            width=20, alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Mom 滾動 10 年年化溢酬（%）', fontsize=12)
axes[1].set_ylabel('年化溢酬（%）')

plt.tight_layout()
plt.show()

## 七、四因子相關性

In [ ]:
import matplotlib.colors as mcolors

corr = factors[['Mkt-RF','SMB','HML','Mom']].corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

labels = ['市場 (Mkt-RF)', '規模 (SMB)', '價值 (HML)', '動能 (Mom)']
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(labels, fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                fontsize=11, color='white' if abs(corr.iloc[i,j]) > 0.5 else 'black')

ax.set_title('四因子相關係數矩陣', fontsize=12)
plt.tight_layout()
plt.show()

print("\n觀察：")
print(f"• Mkt-RF 與 SMB 相關性：{corr.loc['Mkt-RF','SMB']:.2f}")
print(f"• SMB 與 HML 相關性：{corr.loc['SMB','HML']:.2f}")
print(f"• HML 與 Mom 相關性：{corr.loc['HML','Mom']:.2f}（通常為負！）")

## 八、討論

**結論：三個因子溢酬（規模、價值、動能）在長達近百年的美股資料中確實存在。**

**思考問題：**
1. 如果這些溢酬是公開的學術研究結論，為什麼沒有被套利消失？
2. 價值溢酬在 2010 年後大幅萎縮。是因子失效了，還是只是暫時的週期？
3. 動能和均值回歸同時成立，差別只在時間尺度。你要怎麼用這個知識設計策略？
4. Fama-French 模型能解釋台股嗎？台股的小型股效應是否更明顯？

## 八.五｜台灣本土應用：0050 vs 主動基金

**核心問題：** 既然市場因子溢酬確實存在，直接持有市場有意義嗎？

台灣有一個完美的實驗案例——元大台灣50（0050），2003 年成立，追蹤台灣市值前 50 大公司：

| 比較項目 | 0050 ETF | 一般主動基金 |
|----------|----------|------------|
| 費用率（管理費） | 0.46% | 1.5–2.0% |
| 選股策略 | 追蹤指數（市場因子） | 主動選股 |
| 透明度 | 完全公開 | 黑箱 |
| 需要基金經理人判斷 | 不需要 | 需要 |

**S&P SPIVA 台灣報告**每年評估主動基金 vs 指數績效：越長期，主動基金輸給指數的比例越高。

這正是 Fama-French 研究的現實推論：
> **市場因子溢酬是最穩定的——被動持有市場，是最有效率取得它的方式。**

In [ ]:
%pip install -q yfinance

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 下載 0050 資料 ─────────────────────────────────────────────────────────────
try:
    raw = yf.download('0050.TW', start='2003-10-01', progress=False, auto_adjust=True)
    price_0050 = raw['Close'].dropna()
    if hasattr(price_0050.columns, '__len__'):   # MultiIndex fallback
        price_0050 = price_0050.iloc[:, 0]
    got_live = len(price_0050) > 100
except Exception as e:
    got_live = False

if got_live:
    print(f"✅ 0050 資料：{price_0050.index[0].date()} 至 {price_0050.index[-1].date()}，共 {len(price_0050)} 個交易日")
else:
    print("⚠️  使用近似年報酬數據（若要下載即時數據請執行 pip install yfinance）")

# ── SPIVA 台灣主動/被動基準報告（S&P 歷年平均） ────────────────────────────────
spiva_horizons    = ['1 年', '3 年', '5 年', '10 年']
spiva_underperform = [55, 68, 76, 83]   # % 主動基金跑輸大盤

# ── 費用複利侵蝕模擬（初始 100 元，年報酬 8%）────────────────────────────────
initial    = 100
years      = np.arange(0, 21)
mkt_ret    = 0.08
fee_etf    = 0.0046   # 0050 費用率
fee_active = 0.0175   # 主動基金平均費用率

val_etf    = initial * (1 + mkt_ret - fee_etf)    ** years
val_active = initial * (1 + mkt_ret - fee_active) ** years
val_market = initial * (1 + mkt_ret)              ** years

# ── 繪圖 ───────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))

# 圖1：0050 累積報酬（上方全寬）
ax1 = fig.add_subplot(2, 2, (1, 2))
if got_live:
    cum = (price_0050 / price_0050.iloc[0] * 100).squeeze()
    ax1.plot(cum.index, cum.values, color='royalblue', linewidth=1.5)
    ax1.fill_between(cum.index, 100, cum.values, where=(cum.values >= 100),
                     alpha=0.15, color='royalblue')
    n_yr   = (price_0050.index[-1] - price_0050.index[0]).days / 365.25
    ann    = (float(price_0050.iloc[-1]) / float(price_0050.iloc[0])) ** (1/n_yr) - 1
    total  = float(price_0050.iloc[-1]) / float(price_0050.iloc[0]) - 1
    title  = (f'元大台灣50（0050）累積報酬（2003 年成立至今）\n'
              f'總報酬 {total*100:.0f}%，年化報酬 {ann*100:.1f}%，費用率僅 0.46%')
else:
    # 近似年報酬（手動整理，2004–2024）
    approx_yr = list(range(2004, 2025))
    approx_r  = [7.5, 6.1, 30.1, 8.7, -43.1, 73.0, 12.0, -16.6,
                 10.7, 11.8, 14.7, 15.7, -2.4, 29.4, 25.1, -22.0,
                 25.1, 28.9, 10.7, -0.7, 31.4]
    cum_v = [100]
    for r in approx_r:
        cum_v.append(cum_v[-1] * (1 + r/100))
    ax1.plot(approx_yr + [2025], cum_v, color='royalblue', linewidth=1.5,
             marker='o', markersize=3)
    title = '元大台灣50（0050）近似累積報酬（2003 年成立，費用率 0.46%）'

ax1.axhline(100, color='black', linewidth=0.5, linestyle='--')
ax1.set_title(title, fontsize=11)
ax1.set_ylabel('相對 100 的累積價值')

# 圖2：SPIVA 主動基金跑輸比例（左下）
ax2 = fig.add_subplot(2, 2, 3)
colors_s = ['#ff8080' if v > 50 else '#80cc80' for v in spiva_underperform]
bars = ax2.bar(spiva_horizons, spiva_underperform, color=colors_s,
               width=0.5, edgecolor='white', linewidth=1.5)
ax2.axhline(50, color='gray', linewidth=1.2, linestyle='--', label='50% 基準線')
for bar, val in zip(bars, spiva_underperform):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 1.5,
             f'{val}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 100)
ax2.set_ylabel('主動基金跑輸指數的比例 (%)')
ax2.set_title('SPIVA 台灣報告：主動基金跑輸大盤比例\n（S&P 全球指數委員會，歷年平均）', fontsize=10)
ax2.legend(fontsize=9)

# 圖3：費用侵蝕複利（右下）
ax3 = fig.add_subplot(2, 2, 4)
ax3.plot(years, val_market, 'k--', linewidth=1, alpha=0.5, label='無費用（市場報酬）')
ax3.plot(years, val_etf,    color='royalblue', linewidth=2.5,
         label=f'0050 ETF（費用率 0.46%）')
ax3.plot(years, val_active, color='tomato', linewidth=2.5,
         label=f'主動基金（費用率 1.75%）')

gap = val_etf[-1] - val_active[-1]
mid = (val_etf[-1] + val_active[-1]) / 2
ax3.annotate(f'20 年後差距\n≈ {gap:.0f} 元', xy=(20, mid),
             xytext=(14.5, mid + 30), fontsize=10, ha='center',
             arrowprops=dict(arrowstyle='->', color='black', lw=1))
ax3.set_title('費用對複利的侵蝕\n（初始 100 元，假設市場年報酬 8%）', fontsize=10)
ax3.set_xlabel('投資年數')
ax3.set_ylabel('資產價值（元）')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

# 數字摘要
print(f"\n20 年後複利結果（每投入 100 元）：")
print(f"  無費用基準：     {val_market[-1]:.0f} 元")
print(f"  0050（0.46%）：  {val_etf[-1]:.0f} 元  ← 費用率低，保留 {val_etf[-1]/val_market[-1]*100:.1f}% 的市場報酬")
print(f"  主動基金（1.75%）：{val_active[-1]:.0f} 元  ← 費用侵蝕，僅保留 {val_active[-1]/val_market[-1]*100:.1f}% 的市場報酬")
print(f"\n  ETF vs 主動基金 20 年差距：{gap:.0f} 元（{gap/val_active[-1]*100:.0f}% 差距）")
print(f"\n→ 光是費用差距，20 年後就差了將近 {gap:.0f}%——")
print(f"  主動基金需要持續每年多賺 {(fee_active - fee_etf)*100:.2f}% 才能打平費用差距")

## 九、這跟你有什麼關係？

**因子溢酬存在，但費用會把它吃掉。**

這個研究最大的個人投資含義不是「去買小型價值股」，而是：

> 市場風險溢酬（Mkt-RF）是最穩定的因子，年化約 8%。
> 你只需要買一支費用率 0.1% 的全市場 ETF，就能拿到幾乎全部的市場溢酬。
> 主動選股試圖多拿 SMB/HML，但大多數人的交易成本和錯誤決策比這些溢酬還貴。

In [ ]:
mkt_premium = factors['Mkt-RF'].mean() * 12 * 100   # 年化市場溢酬

products = [
    ("全市場 ETF（VOO/VT）",          0.03),
    ("台灣 0050 ETF",                  0.43),
    ("主動型基金（市場平均）",          1.50),
    ("高費用主動基金",                  2.50),
]

print(f"市場因子歷史年化溢酬：{mkt_premium:.1f}%")
print()
print(f"{'投資工具':^24} | {'年費用率':>8} | {'實際拿到':>10} | {'溢酬保留率':>10}")
print("-" * 60)
for name, fee in products:
    net  = mkt_premium - fee
    pct  = net / mkt_premium * 100
    print(f"  {name:^24} | {fee:>7.2f}% | {net:>9.1f}% | {pct:>9.0f}%")

print()
print("結論：選股能力要能持續超越市場 1.5% 以上，才有理由買主動型基金")
print("學術研究顯示，能長期做到這件事的主動基金，不到 10%")
print("對一般投資人而言，'不要輸給市場' 比 '打敗市場' 容易得多")

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| SMB（規模） | 小公司長期溢酬存在，但波動大、近年縮小 |
| HML（價值） | 高 Book/Market 公司長期超額報酬 |
| 動能（Mom） | 強勢股繼續強勢（3–12個月）；夏普比率是所有因子最高之一 |
| 因子溢酬穩定性 | 任何因子都可能連續10年表現不佳 |
| 台灣本土驗證 | 0050 ETF 20 年長期績效優於 76%+ 的台灣主動式股票基金（SPIVA） |
| 費用的力量 | 費用率差距 1.3% 看似微小，20 年後複利差距超過 20%+ |
| 個人應用 | 市場因子最穩定——低費用 ETF（0050、VT）是取得它最有效率的工具 |

> **下一章：** 五因子模型——品質與保守投資的超額報酬